# SAGE-Lite B2: Phase 5.1 — SAGE Learning Rate Isolation Sweep
**Architecture:** Canonical D4 K2 (ConvNeXtV2-Femto + ViT-Tiny 4 layers, P3-C ASDW, top_k = 2, router_hidden_dim = 64)
**Branch:** `crack500-audit` | **Hardware Target:** Google Colab Tesla T4 (16GB VRAM)

### Protocol Overview:
- **Baseline 1e-4 (Reuse):** Đã hoàn tất (`P3_C_D4_K2_best_model_b2_global.pth`, Peak S2 Val Dice = **0.7618**, Loss 0.9518, IoU 0.6386). Không cần train lại.
- **Candidate A (`sage_lr = 5e-5`):** `configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr5e5.yaml`
- **Candidate B (`sage_lr = 2e-4`):** `configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr2e4.yaml`
- **Lifecycle:** Train $\to$ Routing Diagnostics $\to$ Error Analysis $\to$ Zip & Export $\to$ Comparative Summary.

## 1. Setup Environment & Repository Synchronization

In [ ]:
# Cell 1: Hardware & CUDA Verification
!nvidia-smi

In [ ]:
# Cell 2: Google Drive Mount & Repository Synchronization
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if not os.path.exists('sage_lite'):
    !git clone -b crack500-audit https://github.com/bach0823/SAGE_LITE.git sage_lite
else:
    %cd /content/sage_lite
    !git fetch origin crack500-audit
    !git checkout crack500-audit
    !git pull origin crack500-audit

%cd /content/sage_lite
!pip install -q -e .
!pip install -q albumentations timm scipy scikit-learn seaborn matplotlib pandas

In [ ]:
# Cell 3: Dataset Invariant Verification
import os

dataset_root = '/content/dataset/Crack500'
if not os.path.exists(dataset_root):
    print(f'[Dataset] {dataset_root} not found! Extracting/preparing Crack500...')
    !python prepare_data/prepare_crack500.py
else:
    print(f'[Dataset] Found canonical dataset at {dataset_root}')

assert os.path.exists(os.path.join(dataset_root, 'train', 'images')), 'Train images missing!'
assert os.path.exists(os.path.join(dataset_root, 'val', 'images')), 'Val images missing!'
print('[Dataset] Verification PASS: Train and Val splits confirmed ready.')

## 2. CANDIDATE A: SAGE LR = 5e-5

In [ ]:
# Cell 4: Train Candidate A (sage_lr = 5e-5)
%cd /content/sage_lite
!python scripts/train_crack.py --config configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr5e5.yaml --two-stage

In [ ]:
# Cell 5: Diagnose Candidate A (Routing Diagnostics + Error Analysis)
%cd /content/sage_lite
import os

cand_a_dir = "/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5"
cand_a_ckpt = os.path.join(cand_a_dir, "best_model_b2_global.pth")
cand_a_diag_val = os.path.join(cand_a_dir, "diagnostics", "full_val")
cand_a_diag_err = os.path.join(cand_a_dir, "diagnostics", "error_analysis")
cand_a_cfg = "configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr5e5.yaml"

print("[Candidate A Diagnostics] 1/2: Running analyze_routing.py (Routing Statistics)...\n")
!python tools/analyze_routing.py \
    --config {cand_a_cfg} \
    --checkpoint {cand_a_ckpt} \
    --output_dir {cand_a_diag_val} \
    --split val

print("\n[Candidate A Diagnostics] 2/2: Running run_p3_c_error_analysis.py (Error Analysis & Morphology)...\n")
!python tools/run_p3_c_error_analysis.py \
    --config {cand_a_cfg} \
    --checkpoint {cand_a_ckpt} \
    --routing-json {cand_a_diag_val}/routing_statistics.json \
    --output-dir {cand_a_diag_err} \
    --data-root /content/dataset/Crack500

print("\n[Candidate A Diagnostics] COMPLETED SUCCESSFULLY!")

In [ ]:
# Cell 6: Zip & Export Candidate A Artifacts
%cd /content/drive/MyDrive/crack_seg
!zip -r P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5_Full.zip P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5
print("\n[Archive] Successfully generated: /content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5_Full.zip")

# Optional: Tự động tải về máy nếu cần
# from google.colab import files
# files.download('/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5_Full.zip')

## 3. CANDIDATE B: SAGE LR = 2e-4

In [ ]:
# Cell 7: Train Candidate B (sage_lr = 2e-4)
%cd /content/sage_lite
!python scripts/train_crack.py --config configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr2e4.yaml --two-stage

In [ ]:
# Cell 8: Diagnose Candidate B (Routing Diagnostics + Error Analysis)
%cd /content/sage_lite
import os

cand_b_dir = "/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4"
cand_b_ckpt = os.path.join(cand_b_dir, "best_model_b2_global.pth")
cand_b_diag_val = os.path.join(cand_b_dir, "diagnostics", "full_val")
cand_b_diag_err = os.path.join(cand_b_dir, "diagnostics", "error_analysis")
cand_b_cfg = "configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr2e4.yaml"

print("[Candidate B Diagnostics] 1/2: Running analyze_routing.py (Routing Statistics)...\n")
!python tools/analyze_routing.py \
    --config {cand_b_cfg} \
    --checkpoint {cand_b_ckpt} \
    --output_dir {cand_b_diag_val} \
    --split val

print("\n[Candidate B Diagnostics] 2/2: Running run_p3_c_error_analysis.py (Error Analysis & Morphology)...\n")
!python tools/run_p3_c_error_analysis.py \
    --config {cand_b_cfg} \
    --checkpoint {cand_b_ckpt} \
    --routing-json {cand_b_diag_val}/routing_statistics.json \
    --output-dir {cand_b_diag_err} \
    --data-root /content/dataset/Crack500

print("\n[Candidate B Diagnostics] COMPLETED SUCCESSFULLY!")

In [ ]:
# Cell 9: Zip & Export Candidate B Artifacts
%cd /content/drive/MyDrive/crack_seg
!zip -r P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4_Full.zip P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4
print("\n[Archive] Successfully generated: /content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4_Full.zip")

# Optional: Tự động tải về máy nếu cần
# from google.colab import files
# files.download('/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4_Full.zip')

## 4. CANDIDATE C: SAGE LR = 3e-4
Higher learning rate candidate for SAGE routers to test peak capacity before plateau.

In [ ]:
# Cell 10: Train Candidate C (sage_lr = 3e-4)
%cd /content/sage_lite
!python scripts/train_crack.py --config configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr3e4.yaml --two-stage

In [ ]:
# Cell 11: Diagnose Candidate C (Routing Diagnostics + Error Analysis)
%cd /content/sage_lite
import os

cand_c_dir = "/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4"
cand_c_ckpt = os.path.join(cand_c_dir, "best_model_b2_global.pth")
cand_c_diag_val = os.path.join(cand_c_dir, "diagnostics", "full_val")
cand_c_diag_err = os.path.join(cand_c_dir, "diagnostics", "error_analysis")
cand_c_cfg = "configs/p3_ablation/b2_p3_run_c_d4_k2_h64_phase5_sagelr3e4.yaml"

print("[Candidate C Diagnostics] 1/2: Running analyze_routing.py (Routing Statistics)...\n")
!python tools/analyze_routing.py \
    --config {cand_c_cfg} \
    --checkpoint {cand_c_ckpt} \
    --output_dir {cand_c_diag_val} \
    --split val

print("\n[Candidate C Diagnostics] 2/2: Running run_p3_c_error_analysis.py (Error Analysis & Morphology)...\n")
!python tools/run_p3_c_error_analysis.py \
    --config {cand_c_cfg} \
    --checkpoint {cand_c_ckpt} \
    --routing-json {cand_c_diag_val}/routing_statistics.json \
    --output-dir {cand_c_diag_err} \
    --data-root /content/dataset/Crack500

print("\n[Candidate C Diagnostics] COMPLETED SUCCESSFULLY!")

In [ ]:
# Cell 12: Zip & Export Candidate C Artifacts
%cd /content/drive/MyDrive/crack_seg
!zip -r P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4_Full.zip P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4
print("\n[Archive] Successfully generated: /content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4_Full.zip")

# Optional: Tự động tải về máy nếu cần
# from google.colab import files
# files.download('/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4_Full.zip')

## 5. Phase 5.1 Comparative Summary Table

In [ ]:
# Cell 13: Comparative Summary Table for Phase 5.1 Decision Gate
import json
import os
import torch
import pandas as pd

runs = {
    'Candidate A (sage_lr=5e-5)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR5e-5/diagnostics/error_analysis/error_summary.json'
    },
    'Baseline (sage_lr=1e-4)': {
        'dice': 0.7618, 'loss': 0.9518, 'iou': 0.6386, 'status': 'Canonical Baseline (Reused)'
    },
    'Candidate B (sage_lr=2e-4)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR2e-4/diagnostics/error_analysis/error_summary.json'
    },
    'Candidate C (sage_lr=3e-4)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H64_Phase5_SAGELR3e-4/diagnostics/error_analysis/error_summary.json'
    }
}

summary = []
for name, data in runs.items():
    if name == 'Baseline (sage_lr=1e-4)':
        summary.append({'Configuration': name, 'SAGE LR': '1e-4', 'Best Val Dice': 0.7618, 'Best Val Loss': 0.9518, 'Val IoU': 0.6386, 'Status': 'Canonical Baseline'})
    else:
        if '5e-5' in name:
            lr_val = '5e-5'
        elif '2e-4' in name:
            lr_val = '2e-4'
        else:
            lr_val = '3e-4'
        row = {'Configuration': name, 'SAGE LR': lr_val, 'Best Val Dice': None, 'Best Val Loss': None, 'Val IoU': None, 'Status': 'Pending / Missing'}
        if os.path.exists(data['ckpt']):
            c = torch.load(data['ckpt'], map_location='cpu', weights_only=False)
            row['Best Val Dice'] = round(float(c.get('best_dice', 0.0)), 4)
            row['Best Val Loss'] = round(float(c.get('best_loss', 0.0)), 4)
            row['Status'] = f"Epoch {c.get('epoch', '?')} (Stage {c.get('stage', '?')})"
        if os.path.exists(data['err_json']):
            with open(data['err_json']) as f:
                ej = json.load(f)
                row['Val IoU'] = round(float(ej.get('overall_metrics', {}).get('mean_iou', 0.0)), 4)
        summary.append(row)

df = pd.DataFrame(summary)
print("\n" + "="*80)
print("PHASE 5.1: SAGE LR ISOLATION COMPARATIVE SUMMARY (Validation 348 Samples)")
print("="*80)
print(df.to_string(index=False))
print("="*80 + "\n")